# Ingest Bronze Data - Load parquet data from MinIO into Bronze tables

This notebook reads parquet files from MinIO (`abfss://processed@statmigfjzs2p.dfs.core.windows.net/retail_source_data/`) and writes them to the
bronze layer tables in Parquet format. The parquet files already contain metadata columns
(`_source_system`, `_ingestion_timestamp`, `_file_name`, `_record_offset`).

Bronze tables must be created first using `create_bronze_tables.ipynb`.

- **Input:** `abfss://processed@statmigfjzs2p.dfs.core.windows.net/retail_source_data/*.parquet`
- **Output:** `unity_catalog.bronze.*` (Parquet format, partitioned by `ingestion_date`)

> In Databricks the `spark` session is pre-defined, so there is no need to build or stop a `SparkSession`.

In [ ]:
from datetime import datetime

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, to_date, to_timestamp

print("=" * 80)
print("  Bronze Data Ingestion")
print("=" * 80)

## 1. Define bronze tables and their source parquet files

In [1]:
bronze_tables = [
    {"name": "transactions_raw", "parquet_file": "transactions_raw.parquet", "database": "bronze"},
    {"name": "transaction_items_raw", "parquet_file": "transaction_items_raw.parquet", "database": "bronze"},
    {"name": "subscriptions_raw", "parquet_file": "subscriptions_raw.parquet", "database": "bronze"},
    {"name": "customer_interactions_raw", "parquet_file": "customer_interactions_raw.parquet", "database": "bronze"},
    {"name": "product_catalog_raw", "parquet_file": "product_catalog_raw.parquet", "database": "bronze"},
    {"name": "inventory_snapshots_raw", "parquet_file": "inventory_snapshots_raw.parquet", "database": "bronze"},
    {"name": "marketing_campaigns_raw", "parquet_file": "marketing_campaigns_raw.parquet", "database": "bronze"},
    {"name": "campaign_events_raw", "parquet_file": "campaign_events_raw.parquet", "database": "bronze"},
]

## 2. Ingest each table

Reads the pipe-delimited parquet, casts the audit columns to their target types, derives the
`ingestion_date` partition column, aligns the column order with the target table, and
overwrite-inserts into the bronze table.

The partition column is named `ingestion_date` (not `_ingestion_date`): Hadoop's default
`PathFilter` treats paths starting with `_` or `.` as hidden and filters them out, which would
cause Hive to report "Input path does not exist" for partition directories.

In [ ]:
print(f"\nStarting ingestion at: {datetime.now()}")
print(f"Total tables to process: {len(bronze_tables)}\n")

total_records = 0
successful_tables = []
failed_tables = []

for idx, table_config in enumerate(bronze_tables, 1):
    table_name = table_config["name"]
    parquet_file = table_config["parquet_file"]
    database = table_config["database"]
    full_table_name = f"unity_catalog.{database}.{table_name}"

    print(f"[{idx}/{len(bronze_tables)}] Processing: {full_table_name}")
    print(f"  Source: abfss://processed@statmigfjzs2p.dfs.core.windows.net/retail_source_data/{parquet_file}")

    try:
        input_path = f"abfss://processed@statmigfjzs2p.dfs.core.windows.net/retail_source_data/{parquet_file}"
        df = spark.read.parquet(path=input_path)
        record_count = df.count()
        print(f"  ✓ Read {record_count} records from CSV")

        df = df.withColumn("_ingestion_timestamp_casted",
                           to_timestamp(col("_ingestion_timestamp"), "yyyy-MM-dd HH:mm:ss"))
        df = df.withColumn("_record_offset_casted", df["_record_offset"].astype("bigint"))
        df = df.drop("_record_offset").withColumnRenamed("_record_offset_casted", "_record_offset") \
            .drop("_ingestion_timestamp").withColumnRenamed("_ingestion_timestamp_casted", "_ingestion_timestamp")

        df = df.withColumn("ingestion_date", to_date(col("_ingestion_timestamp")))

        cols = [c.col_name for c in
                spark.sql(f"SHOW COLUMNS IN {full_table_name}").select("col_name").collect()]
        df = df.select([df[c] for c in cols])

        print(f"  Inserting into table: {full_table_name}")
        df.write.insertInto(full_table_name, overwrite=True)
        print(f"  ✓ Successfully inserted {record_count} records into {full_table_name}\n")

        total_records += record_count
        successful_tables.append(table_name)

    except Exception as e:
        print(f"  ❌ Failed to process {full_table_name}: {str(e)}\n")
        failed_tables.append(table_name)
        continue

## 3. Ingestion summary

In [ ]:
print("=" * 80)
print("  Ingestion Summary")
print("=" * 80)
print(f"Total records ingested: {total_records:,}")
print(f"Successful tables: {len(successful_tables)}/{len(bronze_tables)}")

if successful_tables:
    print("\n✓ Successfully ingested:")
    for table in successful_tables:
        print(f"  - {table}")

if failed_tables:
    print("\n❌ Failed to ingest:")
    for table in failed_tables:
        print(f"  - {table}")

print("\n" + "=" * 80)
if len(failed_tables) == 0:
    print("  SUCCESS! All bronze tables ingested")
    print("=" * 80)
    print("\nView data in MinIO Console: http://localhost:9001 (admin/admin123)")
    print("\nNext steps:")
    print("  - Run create_silver_tables.ipynb to transform data to silver layer")
    print("  - Query tables: spark.sql('SELECT * FROM unity_catalog.bronze.transactions_raw LIMIT 10').show()")
else:
    print("  COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"\n{len(failed_tables)} table(s) failed to ingest. Check logs above.")